# Character-Level Name Generation with RNN

This notebook trains a character-level RNN to generate names, then analyzes how many generated names are novel versus memorized from the training data.

Now we'll import the necessary libraries and enable autoreload so changes to our shared library are automatically loaded.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as L
import wandb

# Import shared utilities from local package
from aiml_notebooks import create_dataset, create_dataloaders
from aiml_notebooks import log_gradients, log_model_weights, log_gradient_flow
from aiml_notebooks import create_trainer

# Enable autoreload for hot reloading of library changes
%load_ext autoreload
%autoreload 2

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")

In [ ]:
# Configuration (Base defaults - can be overridden by papermill parameters)
CONFIG = {
    # Data
    # TODO: use enum
    'dataset_id': 'palindromes',     # Dataset to use: 'names', 'words', 'palindromes', or 'bitflipping' 
    'seed': 42,                      # Random seed for reproducibility
    'train_split': 0.9,              # Fraction of data for training (rest is validation)
    'batch_size': 128,               # Number of examples per training batch
    
    # Model
    'cell_type': 'lstm',             # Type of recurrent cell: 'rnn', 'gru', or 'lstm'
    'embedding_dim': 64,             # Size of character embedding vectors
    'hidden_size': 256,              # Number of units in RNN hidden layers
    'num_layers': 2,                 # Number of stacked RNN layers
    'dropout': 0.2,                  # Dropout rate to prevent overfitting
    'learning_rate': 1e-3,           # Step size for optimizer (0.001)
    
    # Training
    'max_epochs': 100,               # Number of complete passes through training data
    'log_every_n_steps': 20,         # How often to log training metrics
    
    # Generation
    'max_length': 15,                # Maximum characters in generated text
    'temperature': 0.8,              # Sampling randomness (lower=conservative, higher=creative)
    'sample_size': 20,               # Number of examples to generate
    # TODO: what is this for?
    'novelty_sample_size': 100,      # Number of samples for novelty analysis
    
    # TODO: softcode this
    # Weights & Biases
    'wandb_project': 'name-generation-rnn',  # W&B project name
    'wandb_run_name': None,          # Optional run name (None = auto-generated)
}

# Set random seeds
L.seed_everything(CONFIG['seed'])

These papermill parameters allow hyperparameter sweeps to override the default CONFIG values.

In [ ]:
# Create dataset using the factory (handles data loading, tokenization, and splitting)
full_dataset, train_dataset, val_dataset = create_dataset(
    dataset_id=CONFIG['dataset_id'],
    splits=[CONFIG['train_split'], 1 - CONFIG['train_split']]
)

# Extract the tokenizer from the full dataset for later use
tokenizer = full_dataset.tokenizer

print(f"Dataset: {CONFIG['dataset_id']}")
print(f"Total samples: {len(full_dataset)}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Tokenizer: {tokenizer}")

Now we'll use the dataset factory to download the names, create the tokenizer, and split the data.

In [ ]:
# Create data loaders using the factory
train_loader, val_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=CONFIG['batch_size']
)

# Print sample from the dataset to verify data is accurate
print("\nDataset Sample (first 10 examples):")
print("="*50)
sample_texts = full_dataset.get_texts()[:10]
for i, text in enumerate(sample_texts, 1):
    print(f"{i:2}. {text}")
print("="*50)

Now we'll use the dataloader factory to create batched, shuffled loaders for training and validation.

In [ ]:
LAYERS = {
    'rnn': nn.RNN,
    'gru': nn.GRU,
    'lstm': nn.LSTM,
}

# Define the character-level RNN model with embedding, RNN/GRU/LSTM layers, and generation method
class NameGeneratorRNN(L.LightningModule):
    def __init__(
        self, 
        vocab_size: int, 
        embedding_dim: int = 64, 
        hidden_size: int = 256,
        num_layers: int = 2, 
        dropout: float = 0.2, 
        learning_rate: float = 1e-3,
        cell_type: str = 'rnn'
    ):
        super().__init__()

        self.save_hyperparameters()
        
        # Embedding layer to map characters to vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        model_cls = LAYERS[cell_type.lower()]
        self.rnn = model_cls(
            embedding_dim, 
            hidden_size, 
            num_layers, 
            batch_first=True, # TODO: explain
            dropout=dropout if num_layers > 1 else 0 # TODO: explain
        )
        
        # Dropout layer randomly zeroes a percentage of outputs
        # from the RNN layer before passing them to the fully connected layer
        # (regularization to prevent overfitting)
        self.dropout = nn.Dropout(dropout)

        # Fully connected layer to map hidden state to vocabulary size
        self.fc = nn.Linear(hidden_size, vocab_size)

        # Cross-entropy loss function to measure how well the model's predictions match the true labels
        self.criterion = nn.CrossEntropyLoss()
    
    def forward(self, x, hidden=None):
        # Embed the input characters into a dense vector space
        embedded = self.embedding(x)

        # Pass the embedded input through the RNN layer
        # (returns the output and the final hidden state)
        rnn_out, hidden = self.rnn(embedded, hidden)

        # Apply dropout to the RNN output (regularization)
        rnn_out = self.dropout(rnn_out)

        # Map the hidden state to the vocabulary size
        # (produces a logit for each character in the vocabulary)
        logits = self.fc(rnn_out)

        # Return the logits and the final hidden state
        return logits, hidden
    
    def training_step(self, batch, batch_idx):
        # Forward pass to get logits for each character in batch
        x, y = batch
        logits, _ = self(x)

        # Compute the loss between the logits and the true labels
        loss = self.criterion(logits.view(-1, self.hparams.vocab_size), y.view(-1))

        # Log the loss for this batch
        self.log('train_loss', loss, prog_bar=True)
        
        # Log gradients every N steps (reduce overhead)
        if batch_idx % 10 == 0: # TODO: softcode logging frequency
            log_gradients(self, step=self.global_step) # TODO: what does this do?
        
        # Return the loss for this batch
        # (Lightning will perform backpropagation and step the optimizer)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch

        # Forward pass to get logits for each character in batch
        logits, _ = self(x)

        # Compute the loss between the logits and the true labels
        loss = self.criterion(logits.view(-1, self.hparams.vocab_size), y.view(-1)) # TODO: encapsulate this

        # Log the validation loss for this batch
        self.log('val_loss', loss, prog_bar=True)

        # TODO: why?
        # Return the loss for this batch
        return loss
    
    def on_train_epoch_end(self):
        # Log detailed gradient flow visualization at end of each epoch
        log_gradient_flow(self, step=self.global_step) # TODO: what does this do?
        log_model_weights(self, step=self.global_step) # TODO: what does this do?
    
    def configure_optimizers(self):
        # TODO: softcode optimizer
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
    
    # TODO: should this be separated from model? 
    @torch.no_grad()
    def generate(self, tokenizer, max_length=20, temperature=1.0, num_samples=1):
        """Generate names using the tokenizer."""
        self.eval()
        generated_names = []
        
        for _ in range(num_samples):
            current_idx = tokenizer.get_special_token_idx()
            name_chars = []
            hidden = None
            
            for _ in range(max_length):
                x = torch.tensor([[current_idx]], dtype=torch.long, device=self.device)
                logits, hidden = self(x, hidden)
                probs = F.softmax(logits[0, -1] / temperature, dim=0)
                next_idx = torch.multinomial(probs, 1).item()
                next_char = tokenizer.decode_char(next_idx)
                
                if tokenizer.is_special_token(next_char):
                    break
                
                name_chars.append(next_char)
                current_idx = next_idx
            
            generated_names.append(''.join(name_chars))
        
        return generated_names

# Initialize the model with CONFIG hyperparameters
model = NameGeneratorRNN( # TODO: call this something else
    vocab_size=tokenizer.vocab_size,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_size=CONFIG['hidden_size'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
    learning_rate=CONFIG['learning_rate'],
    cell_type=CONFIG['cell_type'],
)

print(f"Initialized model with {CONFIG['cell_type'].upper()} cell type")

#TODO: update markdown
Now we'll define our RNN model with embedding, recurrent, and output layers, plus a generation method.

In [ ]:
# Train the model (W&B logger created automatically)
trainer = create_trainer(
    max_epochs=CONFIG['max_epochs'],
    log_every_n_steps=CONFIG['log_every_n_steps'],
    wandb_project=CONFIG['wandb_project'],
    wandb_run_name=CONFIG['wandb_run_name'],
    wandb_config=CONFIG,
    model=model
)
trainer.fit(model, train_loader, val_loader)

Now we'll train the model using PyTorch Lightning's Trainer with W&B logging.

In [ ]:
# Generate sample names
generated = model.generate(
    tokenizer,
    max_length=CONFIG['max_length'], 
    temperature=CONFIG['temperature'], 
    num_samples=CONFIG['sample_size']
)
generated = [name.capitalize() for name in generated]
print(", ".join(generated))

# Analyze novelty (new vs existing names)
print("\n" + "="*50)
print("NOVELTY ANALYSIS")
print("="*50)

sample = model.generate(
    tokenizer,
    max_length=CONFIG['max_length'], 
    temperature=CONFIG['temperature'], 
    num_samples=CONFIG['novelty_sample_size']
)
sample = [name for name in sample if name]

# Get original texts from the full dataset for comparison
original_texts_set = set(full_dataset.get_texts())
new_names = [name for name in sample if name not in original_texts_set]
existing_names = [name for name in sample if name in original_texts_set]

# Calculate metrics
novelty_pct = len(new_names) / len(sample) * 100
uniqueness_pct = len(set(sample)) / len(sample) * 100

print(f"Total: {len(sample)}, Unique: {len(set(sample))}")
print(f"✨ NEW: {len(new_names)} ({novelty_pct:.1f}%)")
print(f"♻️  EXISTING: {len(existing_names)} ({len(existing_names)/len(sample)*100:.1f}%)")

print(f"\nNEW: {', '.join([n.capitalize() for n in new_names[:10]])}")
print(f"EXISTING: {', '.join([n.capitalize() for n in existing_names[:10]])}")

# Log to W&B
wandb.log({
    'novelty_percentage': novelty_pct,
    'uniqueness_percentage': uniqueness_pct,
    'total_generated': len(sample),
    'new_names_count': len(new_names),
    'memorized_names_count': len(existing_names),
})

# Log sample names as a table
wandb.log({
    'sample_new_names': wandb.Table(
        columns=['name'],
        data=[[n.capitalize()] for n in new_names[:20]]
    )
})